# บทที่ 2 การโปรแกรมเชิงวิทยาศาสตร์และการทำงานที่ทำซ้ำได้
โน้ตบุ๊กประกอบตำรา *วิทยาศาสตร์ข้อมูลและการเรียนรู้ของเครื่องสำหรับนักฟิสิกส์*
โค้ดทุกเซลล์คัดลอกมาจากสคริปต์ในโฟลเดอร์ `ch02_programming/` หัวเซลล์ **โค้ดที่ N.M** ตรงกับเลขในตำรา

**วิธีใช้**
1. รันเซลล์ **เตรียมโค้ด** ด้านล่างก่อนเสมอ (ทุกครั้งที่เปิดโน้ตบุ๊กหรือเริ่ม session ใหม่)
2. แต่ละหัวข้อเริ่มด้วยเซลล์ที่ขึ้นต้นด้วย `%reset -f` ซึ่งล้างตัวแปรเก่า ทำให้ผลเหมือนรันสคริปต์นั้นเดี่ยว ๆ
   ให้รันเซลล์ในหัวข้อเดียวกันเรียงจากบนลงล่าง (Shift+Enter) หรือใช้ Runtime → Run all
3. ใช้หน่วยประมวลผลกลาง (CPU) ตามค่าเริ่มต้น ไม่ต้องเปลี่ยนเป็น GPU
4. ไฟล์ที่สร้างขึ้นอยู่ใน `ch02_programming/outputs/` และหายเมื่อ session จบ ถ้าต้องการเก็บ ให้คัดลอกไปยัง Google Drive

ตัวเลขส่วนใหญ่ควรตรงกับตำรา การคำนวณที่ไวต่อการปัดเศษ (การฝึกโครงข่ายประสาทเทียม
และมอนติคาร์โลในพิกัดต่อเนื่อง) อาจต่างเล็กน้อยเมื่อรันบนเครื่องอื่น ดู `ch02_programming/README.md`


In [ ]:
# เตรียมโค้ด: ดึงโค้ดจาก GitHub (เมื่อรันบน Colab) และติดตั้งเฉพาะไลบรารีที่ยังไม่มี
import os, sys, subprocess, importlib.util
CHAPTER = 'ch02_programming'
REPO = 'https://github.com/busarapa-stack/dsml-for-physicists.git'
if 'google.colab' in sys.modules:
    ROOT = '/content/dsml-for-physicists'
    if not os.path.isdir(ROOT):
        subprocess.run(['git', 'clone', '-q', REPO, ROOT], check=True)
else:                                   # Jupyter บนเครื่องตนเอง: เปิดจากโฟลเดอร์ notebooks/
    ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(os.path.join(ROOT, CHAPTER))
os.makedirs('data', exist_ok=True); os.makedirs('outputs', exist_ok=True)
NEED = {'matplotlib': 'matplotlib', 'nbclient': 'nbclient>=0.8', 'nbformat': 'nbformat>=5.9', 'numpy': 'numpy', 'scipy': 'scipy'}
missing = [req for mod, req in NEED.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
print('โฟลเดอร์ทำงาน:', os.getcwd())
print('ติดตั้งเพิ่ม:', ', '.join(missing) if missing else 'ไม่มี')


## `01_env_check.py` — โค้ดที่ 2.1

Check that the scientific Python stack is installed and runs.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 01_env_check.py  จากโฟลเดอร์ของบท
__file__ = '01_env_check.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Check that the scientific Python stack is installed and runs."""
# Ch.2  physics_env_check.py
#
# Book references: ssec:hello-physics, code:env-check, exercise E2.1
# Expected: every line prints without error and g = 9.80665 m/s^2.
# Version numbers differ from machine to machine; that is normal.
import sys
import numpy
import scipy
import matplotlib
import scipy.constants as const

print(f"Python version : {sys.version.split()[0]}")
print(f"NumPy version  : {numpy.__version__}")
print(f"SciPy version  : {scipy.__version__}")
print(f"Matplotlib ver.: {matplotlib.__version__}")
print()
print(f"g = {const.g} m/s^2 (standard gravity)")
print(f"h = {const.h:.6e} J s")
print(f"c = {const.c} m/s")

## `02_vectorization.py` — โค้ดที่ 2.2, 2.3, 2.4

Ch.2  Loop versus vectorized NumPy, and broadcasting.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 02_vectorization.py  จากโฟลเดอร์ของบท
__file__ = '02_vectorization.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.2  Loop versus vectorized NumPy, and broadcasting.

Book references: ssec:ndarray, code:vec-vs-loop, code:broadcast, code:broadcast-2d
The book says vectorized code can be "tens to hundreds of times" faster for large
arrays. The exact factor depends on the computer; this script measures it.
"""
import time

import numpy as np

SEED = 2026
rng = np.random.default_rng(SEED)

### โค้ดที่ 2.2

In [ ]:
# --- code:vec-vs-loop -------------------------------------------------------
N = 1_000_000
m = rng.uniform(1.0, 2.0, N)          # kg
v = rng.normal(0.0, 1.0, N)           # m/s

t0 = time.perf_counter()
# loop: one particle at a time (avoid)
KE_total = 0.0
for i in range(N):
    KE_total += 0.5 * m[i] * v[i]**2
t1 = time.perf_counter()
KE_loop = KE_total

# vectorized: whole array at once
KE_total = np.sum(0.5 * m * v**2)
t2 = time.perf_counter()
KE_vec = KE_total

print(f"N = {N:,} particles")
print(f"loop       : KE = {KE_loop:.6e} J   time {t1 - t0:.3f} s")
print(f"vectorized : KE = {KE_vec:.6e} J   time {t2 - t1:.4f} s")
print(f"same result (relative difference {abs(KE_loop - KE_vec) / KE_vec:.1e}); "
      f"speed-up on this computer = {(t1 - t0) / (t2 - t1):.0f}x")

# 3D velocities, N x 3 (text after code:vec-vs-loop)
v3 = rng.normal(0.0, 1.0, (N, 3))
KE3 = np.sum(0.5 * m * np.sum(v3**2, axis=1))
print(f"3D: v has shape {v3.shape}; KE = {KE3:.6e} J "
      f"(equipartition check: KE/N = {KE3 / N:.3f} ~ 3/2 * <m> * 1 = {1.5 * m.mean():.3f})")

### โค้ดที่ 2.3

In [ ]:
# --- code:broadcast ---------------------------------------------------------
x0, v0, a = 0.0, 5.0, -9.81
t = np.linspace(0.0, 10.0, 1000)
x = x0 + v0 * t + 0.5 * a * t**2
print(f"\nbroadcast 1D: scalars x0, v0, a with t{t.shape} -> x{x.shape}")

### โค้ดที่ 2.4

In [ ]:
# --- code:broadcast-2d ------------------------------------------------------
v0 = 20.0
theta = np.deg2rad([30, 45, 60])    # shape (3,)
t     = np.linspace(0, 2, 200)      # shape (200,)
y = (v0 * np.sin(theta)[:, None] * t[None, :]
     - 0.5 * 9.81 * t[None, :]**2)
# y.shape == (3, 200): 3 angles x 200 time points
print(f"broadcast 2D: theta{theta.shape}[:, None] with t{t.shape}[None, :] -> y{y.shape}")
assert y.shape == (3, 200)
# the same numbers from an explicit double loop
y_loop = np.array([[v0 * np.sin(th) * tt - 0.5 * 9.81 * tt**2 for tt in t] for th in theta])
print(f"max |y_broadcast - y_loop| = {np.abs(y - y_loop).max():.1e}")

## `03_constants.py` — โค้ดที่ 2.5

Ch.2  Physical constants from scipy.constants, and what 'standard gravity' means.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 03_constants.py  จากโฟลเดอร์ของบท
__file__ = '03_constants.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.2  Physical constants from scipy.constants, and what 'standard gravity' means.

Book references: ssec:scipy-const, code:scipy-const, exercise E2.1 answer
g = 9.80665 m/s^2 is a defined standard value, not the local gravity.
"""
import numpy as np
import scipy.constants as const

print(f"g   = {const.g} m/s^2")
print(f"h   = {const.h:.6e} J s")
print(f"c   = {const.c} m/s")
print(f"e   = {const.e:.6e} C")
print(f"k_B = {const.k:.6e} J/K")
print(f"N_A = {const.Avogadro:.6e} 1/mol")

# every constant carries its unit and uncertainty in physical_constants
for name in ("speed of light in vacuum", "electron mass", "Newtonian constant of gravitation"):
    val, unit, unc = const.physical_constants[name]
    print(f"{name:34s} {val:.10e} {unit:12s} +- {unc:.1e}")


def normal_gravity(lat_deg):
    """International gravity formula (GRS80 form), sea level [m/s^2]."""
    s = np.sin(np.deg2rad(lat_deg))
    s2 = np.sin(np.deg2rad(2 * lat_deg))
    return 9.780327 * (1 + 0.0053024 * s**2 - 0.0000058 * s2**2)


print("\nlocal gravity versus the standard value:")
for place, lat in [("equator", 0.0), ("Bangkok", 13.75), ("Kamphaeng Saen", 14.02),
                   ("pole", 90.0)]:
    gl = normal_gravity(lat)
    print(f"  {place:15s} lat {lat:5.2f} deg : g = {gl:.4f} m/s^2 "
          f"({100 * (gl - const.g) / const.g:+.2f} % vs standard)")

## `04_projectile.py` — โค้ดที่ 2.6

Ch.2  Vectorized projectile trajectories for several angles, with sanity checks.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 04_projectile.py  จากโฟลเดอร์ของบท
__file__ = '04_projectile.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.2  Vectorized projectile trajectories for several angles, with sanity checks.

Book references: ssec:projectile, code:projectile
Checks from the text
    v0 = 20 m/s, theta = 45 deg  ->  R = v0^2 / g = 400 / 9.80665 = 40.79 m
    15/75 deg and 30/60 deg land at the same distance (sin 2θ = sin(180° − 2θ))
"""
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import g

OUT = Path(__file__).resolve().parent / "outputs"
OUT.mkdir(exist_ok=True)


def projectile(v0, theta_deg, n=1000):
    theta = np.deg2rad(theta_deg)
    T = 2.0 * v0 * np.sin(theta) / g
    t = np.linspace(0.0, T, n)
    x = v0 * np.cos(theta) * t
    y = v0 * np.sin(theta) * t - 0.5 * g * t**2
    return t, x, y


v0 = 20.0
fig, ax = plt.subplots(figsize=(7, 4))
ranges = {}
for theta in (15, 30, 45, 60, 75):
    _, x, y = projectile(v0, theta)
    ranges[theta] = x[-1]
    ax.plot(x, y, label=f"theta = {theta} deg")
ax.set_xlabel("x (m)"); ax.set_ylabel("y (m)")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "ch02_projectile.png", dpi=150)

R_formula = v0**2 / g
print(f"R(45 deg) from code  = {ranges[45]:.4f} m")
print(f"R = v0^2/g = 400/{g} = {R_formula:.4f} m  -> match: {np.isclose(ranges[45], R_formula)}")
for a, b in ((15, 75), (30, 60)):
    print(f"R({a}) = {ranges[a]:.4f} m, R({b}) = {ranges[b]:.4f} m  -> equal: "
          f"{np.isclose(ranges[a], ranges[b])}")
print("saved outputs/ch02_projectile.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `05_free_fall_solve_ivp.py` — โค้ดที่ 2.7

Ch.2  solve_ivp for free fall, checked against the exact solution.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 05_free_fall_solve_ivp.py  จากโฟลเดอร์ของบท
__file__ = '05_free_fall_solve_ivp.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.2  solve_ivp for free fall, checked against the exact solution.

Book references: ssec:solve-ivp, code:solve-ivp, misconception box on default rtol/atol
"""
import numpy as np
from scipy.constants import g
from scipy.integrate import solve_ivp


def free_fall(t, y):
    x, v = y
    return [v, -g]


y0 = [0.0, 20.0]                     # x = 0 m, v = 20 m/s upward
t_span = (0.0, 5.0)
t_eval = np.linspace(*t_span, 200)

sol = solve_ivp(free_fall, t_span, y0, t_eval=t_eval,
                method='RK45', rtol=1e-8, atol=1e-10)
assert sol.success

x_exact = y0[1] * sol.t - 0.5 * g * sol.t**2
print(f"steps accepted: {sol.t.size} output points, {sol.nfev} function evaluations")
print(f"max |x_num - x_exact| = {np.abs(sol.y[0] - x_exact).max():.1e} m")
print("(a constant acceleration is integrated exactly by RK45, so the error is round-off)")

sol_def = solve_ivp(free_fall, t_span, y0, t_eval=t_eval)   # default rtol=1e-3, atol=1e-6
print(f"with default tolerances: max error = {np.abs(sol_def.y[0] - x_exact).max():.1e} m, "
      f"{sol_def.nfev} evaluations")
print("-> for free fall the defaults are fine; 06_oscillator_energy.py shows a case where they are not")

## `06_oscillator_energy.py` — โค้ดที่ 2.8, 2.9

Ch.2  Undamped and damped harmonic oscillator; energy as a correctness check.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 06_oscillator_energy.py  จากโฟลเดอร์ของบท
__file__ = '06_oscillator_energy.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.2  Undamped and damped harmonic oscillator; energy as a correctness check.

Book references: ssec:ho, ssec:energy-analysis, code:ho, code:energy-fit,
                 misconception boxes "the default tolerances of solve_ivp are always
                 enough" and "the energy of a damped oscillator decays exactly exponentially"
The two listings are reproduced line for line below; the extra checks follow.
Text claims checked
    rtol=1e-9: relative energy drift of the undamped case ~ 3e-8
    fitted gamma printed as 0.1001 (set 0.10), equal to three decimals
    default rtol/atol over 20 periods: energy varies by about 0.35 %
"""
import numpy as np
from scipy.integrate import solve_ivp

### โค้ดที่ 2.8

In [ ]:
# --- code:ho ----------------------------------------------------------------
def ho_rhs(t, y, omega, gamma):
    x, v = y
    return [v, -2.0 * gamma * v - omega**2 * x]

omega = 2.0 * np.pi
y0 = [1.0, 0.0]
t_span = (0.0, 20.0)
t_eval = np.linspace(*t_span, 4000)

sol_un = solve_ivp(ho_rhs, t_span, y0, t_eval=t_eval,
                   args=(omega, 0.0),
                   method='RK45', rtol=1e-9, atol=1e-11)
sol_d  = solve_ivp(ho_rhs, t_span, y0, t_eval=t_eval,
                   args=(omega, 0.1),
                   method='RK45', rtol=1e-9, atol=1e-11)
assert sol_un.success and sol_d.success

### โค้ดที่ 2.9

In [ ]:
# --- code:energy-fit --------------------------------------------------------
from scipy.optimize import curve_fit

def energy(sol, omega):
    x, v = sol.y[0], sol.y[1]
    return 0.5 * v**2 + 0.5 * omega**2 * x**2

E_un = energy(sol_un, omega)
E_d  = energy(sol_d,  omega)

drift = (E_un.max() - E_un.min()) / E_un[0]
print(f"Undamped: relative energy drift = {drift:.2e}")

def exp_decay(t, A, k):
    return A * np.exp(-k * t)
popt, _ = curve_fit(exp_decay, sol_d.t, E_d, p0=[E_d[0], 0.2])
gamma_fit = popt[1] / 2
print(f"Damped: gamma_set = 0.10, gamma_fit = {gamma_fit:.4f}")

# --- misconception box: default tolerances ---------------------------------
sol_def = solve_ivp(ho_rhs, t_span, y0, t_eval=t_eval, args=(omega, 0.0))  # rtol=1e-3, atol=1e-6
E_def = energy(sol_def, omega)
print(f"\nDefault rtol=1e-3, atol=1e-6, 20 periods: energy varies by "
      f"{100 * (E_def.max() - E_def.min()) / E_def[0]:.2f} %")
print(f"rtol=1e-9, atol=1e-11,        20 periods: energy varies by {100 * drift:.1e} %")
print(f"function evaluations: {sol_def.nfev} (default) vs {sol_un.nfev} (tight)")

# --- misconception box: energy does not decay as a smooth exponential -------
ratio = E_d / exp_decay(sol_d.t, *popt)
print(f"\nDamped E(t) / fitted exponential ranges from {ratio.min():.3f} to {ratio.max():.3f}"
      " -> small steps: energy is lost fastest where |v| is largest")

# --- the shared module oscillator.py (used by 08 and the exercises) gives the same numbers
from oscillator import energy as energy_mod, simulate
assert np.allclose(energy_mod(simulate(0.1)), E_d)
print("oscillator.py reproduces these results exactly")

## `07_noise.py` — โค้ดที่ 2.10

Ch.2  Adding Gaussian noise with a fixed random seed (scipy.stats + Generator).

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 07_noise.py  จากโฟลเดอร์ของบท
__file__ = '07_noise.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.2  Adding Gaussian noise with a fixed random seed (scipy.stats + Generator).

Book references: ssec:noise, code:noise
"""
import numpy as np
from scipy.constants import g
from scipy.stats import norm


def projectile(v0, theta_deg, n=1000):
    theta = np.deg2rad(theta_deg)
    T = 2.0 * v0 * np.sin(theta) / g
    t = np.linspace(0.0, T, n)
    return t, v0 * np.cos(theta) * t, v0 * np.sin(theta) * t - 0.5 * g * t**2


rng = np.random.default_rng(seed=42)
sigma_y = 0.02                        # m

t, x, y_clean = projectile(v0=20.0, theta_deg=45)
noise = norm.rvs(scale=sigma_y, size=y_clean.size, random_state=rng)
y_observed = y_clean + noise

print(f"{y_clean.size} points, sigma_y = {sigma_y} m")
print(f"sample std of the noise = {noise.std(ddof=1):.5f} m (expected {sigma_y})")
print(f"first three noise values: {np.round(noise[:3], 5)}")

# same seed -> identical noise; this is what makes the analysis reproducible
rng2 = np.random.default_rng(seed=42)
noise2 = norm.rvs(scale=sigma_y, size=y_clean.size, random_state=rng2)
print(f"re-created with seed 42 -> identical: {np.array_equal(noise, noise2)}")

## `08_figures.py` — โค้ดที่ 2.11, 2.12, 2.13, 2.14

Ch.2  Matplotlib figures: object-oriented basics, error bars, shaded bands,

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 08_figures.py  จากโฟลเดอร์ของบท
__file__ = '08_figures.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.2  Matplotlib figures: object-oriented basics, error bars, shaded bands,
multi-panel figure, and saving for publication.

Book references: sec:matplotlib, code:mpl-oo, code:errorbar, code:fillbetween,
                 code:multipanel, tab:fig-format
The book shows only the plotting lines of code:errorbar and code:fillbetween;
the data they use (t_obs, x_obs, sigma, t_dense, ...) are created here.
"""
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

from oscillator import GAMMA, OMEGA, energy, simulate

SEED = 2026
OUT = Path(__file__).resolve().parent / "outputs"
OUT.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)
omega = OMEGA

### โค้ดที่ 2.11

In [ ]:
# --- code:mpl-oo ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 4))
t = np.linspace(0, 2*np.pi, 200)
ax.plot(t, np.sin(t), label=r'$\sin(t)$')
ax.plot(t, np.cos(t), label=r'$\cos(t)$', linestyle='--')
ax.set_xlabel('t (rad)')
ax.set_ylabel('amplitude')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
fig.savefig(OUT / "ch02_mpl_oo.png", dpi=150); plt.close(fig)

### โค้ดที่ 2.12

In [ ]:
# --- code:errorbar : few points, one error bar each -------------------------
sigma = 0.08                                       # m
t_obs = np.linspace(0, 2, 21)                      # 21 measurements
x_obs = np.cos(omega * t_obs) + rng.normal(0, sigma, t_obs.size)
t_fine = np.linspace(0, 2, 500)

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(t_obs, x_obs, yerr=sigma,
            fmt='o', capsize=3, label='measured', color='C0')
ax.plot(t_fine, np.cos(omega*t_fine),
        label=r'theory: $\cos(\omega t)$', color='C3')
ax.legend(); ax.grid(alpha=0.3)
ax.set_xlabel('t (s)'); ax.set_ylabel('x (m)')
ax.set_title(r'error bars = $\pm 1\sigma$ of each measurement ($\sigma$ = 0.08 m)')
fig.tight_layout(); fig.savefig(OUT / "ch02_errorbar.png", dpi=150); plt.close(fig)
inside = np.abs(x_obs - np.cos(omega * t_obs)) < sigma
n_exp = 0.683 * t_obs.size
n_sd = np.sqrt(t_obs.size * 0.683 * 0.317)          # binomial spread
print(f"errorbar: {t_obs.size} points, {inside.sum()} within 1 sigma of theory "
      f"(expected {n_exp:.0f} +- {n_sd:.0f}; with only 21 points this count fluctuates a lot)")

### โค้ดที่ 2.13

In [ ]:
# --- code:fillbetween : dense data, shaded band -----------------------------
t_dense = np.linspace(0, 2, 2000)
x_dense_true = np.cos(omega * t_dense)
sigma_band = 0.05 + 0.05 * t_dense / 2             # noise that grows with time
x_dense_obs = x_dense_true + rng.normal(0, sigma_band)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(t_dense, x_dense_obs, color='C0', lw=0.6, label='data')
ax.plot(t_dense, x_dense_true, color='C3', lw=1.5, label='theory')
ax.fill_between(t_dense,
                x_dense_true - sigma_band,
                x_dense_true + sigma_band,
                alpha=0.3, color='C3', label=r'$\pm\sigma(t)$')
ax.legend(); ax.grid(alpha=0.3)
ax.set_xlabel('t (s)'); ax.set_ylabel('x (m)')
fig.tight_layout(); fig.savefig(OUT / "ch02_fillbetween.png", dpi=150); plt.close(fig)
frac = np.mean(np.abs(x_dense_obs - x_dense_true) < sigma_band)
print(f"fill_between: {t_dense.size} points, {100 * frac:.1f} % inside the ±sigma band")

### โค้ดที่ 2.14

In [ ]:
# --- code:multipanel --------------------------------------------------------
sol_un, sol_d = simulate(0.0), simulate(GAMMA)
fig, axes = plt.subplots(3, 1, figsize=(7, 8), sharex=True)

axes[0].plot(sol_un.t, sol_un.y[0], label='undamped', color='C0')
axes[0].plot(sol_d.t,  sol_d.y[0],  label='damped (gamma = 0.1 1/s)', color='C3')
axes[0].set_ylabel(r'$x(t)$ (m)'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(sol_un.t, sol_un.y[1], color='C0')
axes[1].plot(sol_d.t,  sol_d.y[1],  color='C3')
axes[1].set_ylabel(r'$v(t)$ (m/s)'); axes[1].grid(alpha=0.3)

axes[2].plot(sol_un.t, energy(sol_un, omega), color='C0')
axes[2].plot(sol_d.t,  energy(sol_d,  omega), color='C3')
axes[2].set_ylabel(r'$E(t)/m$ (J/kg)')
axes[2].set_xlabel(r'$t$ (s)'); axes[2].grid(alpha=0.3)

fig.suptitle(r'Harmonic oscillator ($\omega = 2\pi$ rad/s)')
plt.tight_layout()

# --- saving for publication (tab:fig-format) --------------------------------
fig.savefig(OUT / 'ho_comparison.pdf', dpi=300, bbox_inches='tight')   # book: fig.savefig('ho_comparison.pdf', ...)
fig.savefig(OUT / 'ho_comparison.png', dpi=150, bbox_inches='tight')
plt.close(fig)
for f in ("ch02_mpl_oo.png", "ch02_errorbar.png", "ch02_fillbetween.png",
          "ho_comparison.pdf", "ho_comparison.png"):
    print(f"saved outputs/{f:24s} {(OUT / f).stat().st_size / 1024:7.1f} kB")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `09_random_seed.py` — โค้ดที่ 2.19

Ch.2  The modern NumPy random Generator versus the legacy global seed.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python 09_random_seed.py  จากโฟลเดอร์ของบท
__file__ = '09_random_seed.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""Ch.2  The modern NumPy random Generator versus the legacy global seed.

Book references: ssec:seed, code:seed-modern
"""
import numpy as np

# recommended: Generator object
rng = np.random.default_rng(seed=42)
samples = rng.normal(loc=0, scale=1, size=1000)
samples_new = samples

# legacy: hidden global state (avoid in new code)
np.random.seed(42)
samples = np.random.normal(loc=0, scale=1, size=1000)
samples_legacy = samples
samples = samples_new

print(f"Generator(seed=42): mean {samples.mean():+.5f}, first {samples[0]:+.5f}")
print(f"legacy seed(42)   : mean {samples_legacy.mean():+.5f}, first {samples_legacy[0]:+.5f}")
print("-> the same seed number gives DIFFERENT numbers in the two systems;"
      " a seed is only meaningful together with the generator that uses it")


# why the global state is fragile: a library call in between changes your numbers
def analysis_legacy():
    np.random.seed(1)
    a = np.random.normal(size=3)
    _ = np.random.random()                 # e.g. a helper function that also draws
    b = np.random.normal(size=3)
    return b


def analysis_generator():
    rng_data = np.random.default_rng(1)
    rng_other = np.random.default_rng(2)   # the helper gets its own generator
    a = rng_data.normal(size=3)
    _ = rng_other.random()
    b = rng_data.normal(size=3)
    return b


def analysis_generator_no_helper():
    rng_data = np.random.default_rng(1)
    a = rng_data.normal(size=3)
    b = rng_data.normal(size=3)
    return b


print("\nadding one extra random draw in a helper function:")
print(f"  separate Generators: b unchanged = "
      f"{np.array_equal(analysis_generator(), analysis_generator_no_helper())}")
np.random.seed(1); np.random.normal(size=3); b_ref = np.random.normal(size=3)
print(f"  global legacy state: b unchanged = {np.array_equal(analysis_legacy(), b_ref)}")

# แนวคำตอบของแบบฝึกหัดที่ต้องรันโค้ด
ควรลองทำเองก่อน แล้วจึงรันเซลล์เหล่านี้เพื่อเทียบคำตอบ

## `exercises/E2_2_drag.py`

E2.2  Projectile with linear air drag, F = -b m v. Landing point with events.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E2_2_drag.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E2_2_drag.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E2.2  Projectile with linear air drag, F = -b m v. Landing point with events.

Book answer: range about 34.1 m for v0 = 20 m/s, 45 deg, b = 0.1 1/s;
setting b = 0 must give back 40.79 m.
"""
import numpy as np
from scipy.constants import g
from scipy.integrate import solve_ivp


def projectile_drag_rhs(t, y, b):
    x, yy, vx, vy = y
    return [vx, vy, -b * vx, -g - b * vy]


def hit_ground(t, y, b):
    return y[1]


hit_ground.terminal = True        # stop the integration
hit_ground.direction = -1         # only when y goes from + to -


def landing_range(v0=20.0, theta_deg=45.0, b=0.1):
    th = np.deg2rad(theta_deg)
    y0 = [0.0, 0.0, v0 * np.cos(th), v0 * np.sin(th)]
    sol = solve_ivp(projectile_drag_rhs, (0, 10), y0, args=(b,), events=hit_ground,
                    rtol=1e-10, atol=1e-12)
    # the event at t = 0 (y = 0 while rising) is not counted because direction = -1
    t_land = sol.t_events[0][0]
    return sol.y_events[0][0][0], t_land


for b in (0.1, 0.0):
    R, T = landing_range(b=b)
    print(f"b = {b:3.1f} 1/s : range = {R:.2f} m, flight time = {T:.3f} s")
print(f"no-drag formula v0^2/g = {400 / g:.2f} m")

# which angle gives the longest range with drag? (it is below 45 deg)
angles = np.arange(30, 50.5, 0.5)
R = [landing_range(theta_deg=a)[0] for a in angles]
print(f"with b = 0.1 the best angle is {angles[int(np.argmax(R))]:.1f} deg "
      f"(range {max(R):.2f} m), not 45 deg")

## `exercises/E2_3_rtol.py`

E2.3  Energy conservation versus the tolerance rtol, 100 periods, undamped oscillator.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E2_3_rtol.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E2_3_rtol.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E2.3  Energy conservation versus the tolerance rtol, 100 periods, undamped oscillator.

Book answer: the energy error grows with time and shrinks as rtol shrinks;
choose rtol from the plot, not by guessing.
"""
import sys
import time
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from oscillator import energy, simulate

OUT = Path(__file__).resolve().parent.parent / "outputs"
OUT.mkdir(exist_ok=True)

fig, ax = plt.subplots(figsize=(7, 4))
print(" rtol    atol   max|dE/E0| (100 T)  at 10 T    first > 1e-6   evals   time")
for rtol in (1e-3, 1e-6, 1e-9):
    t0 = time.perf_counter()
    sol = simulate(0.0, t_end=100.0, n=20000, rtol=rtol, atol=rtol * 1e-2)
    dt = time.perf_counter() - t0
    E = energy(sol)
    err = np.abs(E - E[0]) / E[0]
    over = sol.t[err > 1e-6]
    first = f"t = {over[0]:6.2f} s" if over.size else "never     "
    print(f"{rtol:5.0e}  {rtol * 1e-2:5.0e}   {err.max():10.2e}        "
          f"{err[sol.t <= 10].max():9.2e}  {first}  {sol.nfev:6d}  {dt:5.2f} s")
    ax.semilogy(sol.t, np.maximum(err, 1e-16), label=f"rtol = {rtol:.0e}")
ax.axhline(1e-6, color="k", ls=":", label="target 1e-6")
ax.set_xlabel("t (s)  [1 period = 1 s]"); ax.set_ylabel(r"$|E - E_0| / E_0$")
ax.legend(); ax.grid(alpha=0.3, which="both")
fig.tight_layout(); fig.savefig(OUT / "E2_3_energy_error.png", dpi=150)
print("-> the error grows about linearly with time; for |dE/E0| <= 1e-6 over 100 periods"
      " only rtol = 1e-9 works (rtol = 1e-6 already fails in the first period)")
print("saved outputs/E2_3_energy_error.png")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E2_4_gamma_fit.py`

E2.4  How well does E(t) = A exp(-2 gamma t) recover gamma as damping grows?

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E2_4_gamma_fit.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E2_4_gamma_fit.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E2.4  How well does E(t) = A exp(-2 gamma t) recover gamma as damping grows?

Simulate for min(8/gamma, 20) s so the energy falls by a similar fraction in each case.
Book answer: gamma = 0.05 and 0.1 -> error <= ~0.1 %; gamma = 0.5 and 2.0 -> a few %.
"""
import sys
from pathlib import Path

import numpy as np
from scipy.optimize import curve_fit

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from oscillator import OMEGA, energy, exp_decay, simulate

print(" gamma   gamma/omega   T_sim   gamma_fit   rel. error")
for gamma in (0.05, 0.1, 0.5, 2.0):
    T = min(8 / gamma, 20.0)
    sol = simulate(gamma, t_end=T)
    E = energy(sol)
    popt, _ = curve_fit(exp_decay, sol.t, E, p0=[E[0], 2 * gamma])
    g_fit = popt[1] / 2
    print(f"{gamma:6.2f}   {gamma / OMEGA:9.4f}   {T:5.1f}   {g_fit:9.5f}   "
          f"{100 * (g_fit - gamma) / gamma:+7.2f} %")
print("-> the exponential is the average trend; it fits well only when gamma << omega (P2b)")

## `exercises/E2_5_error_band.py`

E2.5  Three-panel oscillator figure + Gaussian noise (sigma = 0.03 m, seed 2026),

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E2_5_error_band.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E2_5_error_band.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E2.5  Three-panel oscillator figure + Gaussian noise (sigma = 0.03 m, seed 2026),
shaded ±sigma band around the THEORY curve, and a fourth panel: phase space.
"""
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, str(Path(__file__).resolve().parent.parent))
from oscillator import GAMMA, energy, simulate

OUT = Path(__file__).resolve().parent.parent / "outputs"
OUT.mkdir(exist_ok=True)
rng = np.random.default_rng(seed=2026)       # created ONCE, at the top
sigma = 0.03                                 # m

sol_un, sol_d = simulate(0.0), simulate(GAMMA)
x_obs = sol_d.y[0] + rng.normal(0, sigma, sol_d.t.size)

fig, axes = plt.subplots(4, 1, figsize=(7, 10))
ax = axes[0]
ax.plot(sol_d.t, x_obs, ".", ms=1.5, color="C0", label="damped + noise")
ax.plot(sol_d.t, sol_d.y[0], color="C3", lw=1, label="theory")
ax.fill_between(sol_d.t, sol_d.y[0] - sigma, sol_d.y[0] + sigma, color="C3", alpha=0.3,
                label=r"theory $\pm\sigma$")
ax.set_ylabel(r"$x$ (m)"); ax.legend(loc="upper right"); ax.grid(alpha=0.3)
axes[1].plot(sol_un.t, sol_un.y[1], color="C0"); axes[1].plot(sol_d.t, sol_d.y[1], color="C3")
axes[1].set_ylabel(r"$v$ (m/s)"); axes[1].grid(alpha=0.3)
axes[2].plot(sol_un.t, energy(sol_un), color="C0"); axes[2].plot(sol_d.t, energy(sol_d), color="C3")
axes[2].set_ylabel(r"$E/m$ (J/kg)"); axes[2].set_xlabel("t (s)"); axes[2].grid(alpha=0.3)
axes[3].plot(sol_un.y[0], sol_un.y[1], color="C0", label="undamped: closed ellipse")
axes[3].plot(sol_d.y[0], sol_d.y[1], color="C3", lw=0.8, label="damped: inward spiral")
axes[3].set_xlabel(r"$x$ (m)"); axes[3].set_ylabel(r"$v$ (m/s)")
axes[3].legend(loc="upper right"); axes[3].grid(alpha=0.3)
fig.tight_layout()
fig.savefig(OUT / "ho_with_error_band.pdf", dpi=300, bbox_inches="tight")
inside = np.mean(np.abs(x_obs - sol_d.y[0]) < sigma)
print(f"{100 * inside:.1f} % of the noisy points lie inside the ±sigma band (expected 68.3 %)")
print("saved outputs/ho_with_error_band.pdf")

In [ ]:
# แสดงรูปที่ส่วนนี้สร้าง (สคริปต์บันทึกรูปลงไฟล์ใน outputs/)
from IPython.display import Image, display
import glob, os
for _f in sorted(glob.glob('outputs/**/*.png', recursive=True)):
    if os.path.getmtime(_f) >= float(os.environ['NB_T0']):
        print(_f); display(Image(_f))


## `exercises/E2_10_seed.py`

E2.10  What one seed can and cannot tell you.

In [ ]:
%reset -f
# ทำให้โค้ดทำงานเหมือนสั่ง  python exercises/E2_10_seed.py  จากโฟลเดอร์ของบท
__file__ = 'exercises/E2_10_seed.py'
import os, sys, time; sys.argv = [__file__]; os.environ['NB_T0'] = str(time.time())
_root = os.path.dirname(os.getcwd())         # ล้างโมดูลและ path ของหัวข้อก่อนหน้า (เช่น _shared.py ของบทอื่น)
sys.path[:] = [os.path.dirname(os.path.abspath(__file__))] + [p for p in sys.path if not os.path.abspath(p or '.').startswith(_root + os.sep)]
for _m in [m for m, v in list(sys.modules.items()) if str(getattr(v, '__file__', '') or '').startswith(_root + os.sep)]:
    del sys.modules[_m]
"""E2.10  What one seed can and cannot tell you.

Book answer: same seed -> identical mean; 20 seeds -> means scatter around 0 with
standard deviation about 1/sqrt(1000) = 0.03.
"""
import numpy as np


def mean_of_gaussians(rng, n=1000):
    return rng.normal(0.0, 1.0, n).mean()


a = mean_of_gaussians(np.random.default_rng(7))
b = mean_of_gaussians(np.random.default_rng(7))
print(f"seed 7 twice: {a:+.6f}, {b:+.6f} -> identical: {a == b}")

means = np.array([mean_of_gaussians(np.random.default_rng(s)) for s in range(20)])
print("20 seeds:", " ".join(f"{m:+.3f}" for m in means))
print(f"spread of the 20 means: std = {means.std(ddof=1):.4f}   "
      f"(theory 1/sqrt(1000) = {1 / np.sqrt(1000):.4f})")
print(f"smallest {means.min():+.3f}, largest {means.max():+.3f}: "
      "a single seed could have given any value in this range")

# ตัวอย่างโค้ดที่ไม่ได้รันในโน้ตบุ๊กนี้
เป็นคำสั่งเชลล์ ไฟล์ตั้งค่า โมดูลที่สคริปต์อื่นเรียกใช้ หรือโค้ดสำหรับข้อมูลจริง เปิดดูได้จากแถบไฟล์ของ Colab

| ตัวอย่างโค้ด | ไฟล์ |
|---|---|
| โค้ดที่ 2.15 | `shell/venv.sh` |
| โค้ดที่ 2.16 | `shell/git_flow_demo.sh` |
| โค้ดที่ 2.17 | `shell/gitignore_python.txt` |
| โค้ดที่ 2.18 | `shell/github_push.sh` |